In [ ]:
# CELL 1 — Install & Imports
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "datasets", "scikit-learn", "groq"], check=False)

import os, json, time
import numpy as np
import pandas as pd
from collections import Counter


In [ ]:
# CELL 2 — Label Map
LABEL2ID  = {"analysis": 0, "hot_take": 1, "reaction": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)
print("Labels:", LABEL2ID)


In [ ]:
# CELL 3 — Upload CSV (upload nba_takes_labeled.csv when prompted)
from google.colab import files
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(csv_filename)
print(df["label"].value_counts())
print(f"Total rows: {len(df)}")


In [ ]:
# CELL 4 — Split & Tokenize
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)
print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts, max_length=128):
    return tokenizer(list(texts), padding="max_length", truncation=True,
                     max_length=max_length, return_tensors="pt")

train_enc = tokenize(train_df["text"])
val_enc   = tokenize(val_df["text"])
test_enc  = tokenize(test_df["text"])

train_labels = [LABEL2ID[l] for l in train_df["label"]]
val_labels   = [LABEL2ID[l] for l in val_df["label"]]
test_labels  = [LABEL2ID[l] for l in test_df["label"]]

print("Tokenization complete.")


In [ ]:
# CELL 5 — Groq Zero-Shot Baseline
# Add GROQ_API_KEY to Colab Secrets (key icon in left sidebar) before running
from google.colab import userdata
import groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
client = groq.Groq(api_key=GROQ_API_KEY)

SYSTEM_PROMPT = """You are a classifier for NBA Reddit discourse quality.
Classify each post into exactly one of these three labels:

- analysis: The post makes a structured argument supported by statistics, historical comparisons,
  tactical observations, or verifiable evidence. Evidence is specific and would stand on its own
  even if the opinion framing were removed.
- hot_take: A bold, confident opinion stated without supporting evidence, or with evidence so thin
  or cherry-picked that it functions as decoration rather than reasoning. The post asserts rather
  than argues.
- reaction: An immediate emotional response to a specific recent event. Expresses a feeling in the
  moment with little to no structured argument.

Rules:
1. Reply with ONLY the label name — no punctuation, no explanation, nothing else.
2. The label must be exactly one of: analysis, hot_take, reaction
3. If a post cites a stat but is primarily conclusion-first, label it hot_take.
4. If a post is emotional about a specific moment but mentions a number, label it reaction.
"""

def classify_groq(text):
    try:
        resp = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": text}
            ],
            max_tokens=10, temperature=0
        )
        return resp.choices[0].message.content.strip().lower()
    except Exception as e:
        print(f"Error: {e}")
        return "error"

test_texts = list(test_df["text"])
baseline_preds_raw = []
print(f"Running Groq baseline on {len(test_texts)} test examples...")
for i, text in enumerate(test_texts):
    pred = classify_groq(text)
    baseline_preds_raw.append(pred)
    if (i+1) % 5 == 0:
        print(f"  {i+1}/{len(test_texts)}")
    time.sleep(0.3)
print("Baseline complete.")


In [ ]:
# CELL 6 — Score Baseline
from sklearn.metrics import accuracy_score, f1_score, classification_report

valid_labels = set(LABEL2ID.keys())
baseline_preds, baseline_true = [], []
unparseable = 0

for i, pred in enumerate(baseline_preds_raw):
    if pred in valid_labels:
        baseline_preds.append(LABEL2ID[pred])
        baseline_true.append(test_labels[i])
    else:
        unparseable += 1
        print(f"Unparseable: '{pred}'")

baseline_accuracy = accuracy_score(baseline_true, baseline_preds)
baseline_f1_macro = f1_score(baseline_true, baseline_preds, average="macro")
label_names = [ID2LABEL[i] for i in range(NUM_LABELS)]

print(f"Unparseable: {unparseable}")
print(f"Baseline Accuracy: {baseline_accuracy:.4f}  Macro F1: {baseline_f1_macro:.4f}")
print()
print(classification_report(baseline_true, baseline_preds, target_names=label_names))


In [ ]:
# CELL 7 — Fine-Tune Setup
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

class NBADataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long)
        }

BATCH_SIZE = 16
train_loader = DataLoader(NBADataset(train_enc, train_labels), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(NBADataset(val_enc,   val_labels),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(NBADataset(test_enc,  test_labels),  batch_size=BATCH_SIZE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID).to(device)

LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3
optimizer     = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps   = len(train_loader) * NUM_EPOCHS
warmup_steps  = int(total_steps * 0.1)
scheduler     = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print(f"Epochs: {NUM_EPOCHS} | Steps: {total_steps} | Warmup: {warmup_steps}")


In [ ]:
# CELL 8 — Training Loop (~5-10 min on T4)
def get_preds(loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(input_ids=batch["input_ids"].to(device),
                        attention_mask=batch["attention_mask"].to(device))
            preds.extend(out.logits.argmax(dim=-1).cpu().numpy())
            trues.extend(batch["labels"].numpy())
    return preds, trues

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out  = model(input_ids=batch["input_ids"].to(device),
                     attention_mask=batch["attention_mask"].to(device),
                     labels=batch["labels"].to(device))
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += out.loss.item()
    val_preds, val_true = get_preds(val_loader)
    val_acc = accuracy_score(val_true, val_preds)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  loss={total_loss/len(train_loader):.4f}  val_acc={val_acc:.4f}")

print("Training complete!")


In [ ]:
# CELL 9 — Evaluate Fine-Tuned Model
from sklearn.metrics import confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

test_preds, test_true = get_preds(test_loader)
ft_accuracy = accuracy_score(test_true, test_preds)
ft_f1_macro = f1_score(test_true, test_preds, average="macro")

print(f"Fine-Tuned Accuracy: {ft_accuracy:.4f}  Macro F1: {ft_f1_macro:.4f}")
print()
print(classification_report(test_true, test_preds, target_names=label_names))

cm = confusion_matrix(test_true, test_preds)
print("Confusion Matrix (rows=true, cols=predicted):")
header = f"{'':>12}" + "".join(f"{l:>12}" for l in label_names)
print(header)
for i, row in enumerate(cm):
    print(f"{label_names[i]:>12}" + "".join(f"{v:>12}" for v in row))

# Save PNG
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_xticks(range(NUM_LABELS)); ax.set_xticklabels(label_names, rotation=25, ha="right")
ax.set_yticks(range(NUM_LABELS)); ax.set_yticklabels(label_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Fine-Tuned DistilBERT")
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("Saved confusion_matrix.png")


In [ ]:
# CELL 10 — Wrong Predictions
wrong = [(test_texts[i], ID2LABEL[test_true[i]], ID2LABEL[test_preds[i]])
         for i in range(len(test_true)) if test_true[i] != test_preds[i]]

print(f"Wrong: {len(wrong)} / {len(test_true)}")
print("="*70)
for text, true, pred in wrong:
    print(f"TRUE: {true:10}  PRED: {pred:10}")
    print(f"TEXT: {text[:110]}...")
    print("-"*70)


In [ ]:
# CELL 11 — Sample Classifications with Confidence Scores
import torch.nn.functional as F

model.eval()
sample_enc = tokenizer(test_texts[:5], padding="max_length", truncation=True,
                       max_length=128, return_tensors="pt").to(device)
with torch.no_grad():
    probs = F.softmax(model(**sample_enc).logits, dim=-1)

print(f"{'Text (60 chars)':62} {'Pred':10} {'Conf':8} {'True':10}")
print("-"*95)
for i in range(5):
    pred_id    = probs[i].argmax().item()
    confidence = probs[i][pred_id].item()
    print(f"{test_texts[i][:60]:62} {ID2LABEL[pred_id]:10} {confidence:.1%}   {ID2LABEL[test_true[i]]}")


In [ ]:
# CELL 12 — Side-by-Side Comparison + Export evaluation_results.json
from sklearn.metrics import precision_score, recall_score

def per_class(true, preds):
    result = {}
    for i, lname in enumerate(label_names):
        tb = [1 if t == i else 0 for t in true]
        pb = [1 if p == i else 0 for p in preds]
        result[lname] = {
            "precision": precision_score(tb, pb, zero_division=0),
            "recall":    recall_score(tb, pb, zero_division=0),
            "f1":        f1_score(tb, pb, zero_division=0),
        }
    return result

ft_pc = per_class(test_true, test_preds)
bl_pc = per_class(baseline_true, baseline_preds)

print(f"{'Model':<22} {'Accuracy':>10} {'Macro F1':>10}")
print("="*45)
print(f"{'Groq Zero-Shot':<22} {baseline_accuracy:>10.4f} {baseline_f1_macro:>10.4f}")
print(f"{'DistilBERT Fine-Tuned':<22} {ft_accuracy:>10.4f} {ft_f1_macro:>10.4f}")
print()
print(f"{'Label':<14} {'BL-P':>6} {'BL-R':>6} {'BL-F1':>6}  {'FT-P':>6} {'FT-R':>6} {'FT-F1':>6}")
print("-"*60)
for lname in label_names:
    bl = bl_pc[lname]; ft = ft_pc[lname]
    print(f"{lname:<14} {bl['precision']:>6.3f} {bl['recall']:>6.3f} {bl['f1']:>6.3f}  "
          f"{ft['precision']:>6.3f} {ft['recall']:>6.3f} {ft['f1']:>6.3f}")

results = {
    "baseline": {
        "model": "groq/llama-3.3-70b-versatile (zero-shot)",
        "accuracy": baseline_accuracy, "macro_f1": baseline_f1_macro, "per_class": bl_pc
    },
    "fine_tuned": {
        "model": "distilbert-base-uncased (3 epochs, lr=2e-5, batch=16)",
        "accuracy": ft_accuracy, "macro_f1": ft_f1_macro, "per_class": ft_pc,
        "confusion_matrix": cm.tolist()
    }
}
with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved evaluation_results.json")
print("Download both files from the Files panel (right-click → Download):")
print("  • evaluation_results.json")
print("  • confusion_matrix.png")


In [ ]:
# CELL 13 — Deployed Interface (Stretch Feature) — run separately after Cell 12
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=False)
import gradio as gr
import torch.nn.functional as F

def classify_post(text):
    model.eval()
    enc = tokenizer(text, padding="max_length", truncation=True,
                    max_length=128, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = F.softmax(model(**enc).logits, dim=-1)[0]
    pred_id    = probs.argmax().item()
    confidence = probs[pred_id].item()
    label      = ID2LABEL[pred_id]
    breakdown  = {ID2LABEL[i]: float(probs[i]) for i in range(NUM_LABELS)}
    out  = f"**Label:** `{label}`  |  **Confidence:** {confidence:.1%}\n\n"
    out += "| Label | Probability |\n|-------|-------------|\n"
    for l, p in sorted(breakdown.items(), key=lambda x: -x[1]):
        out += f"| {l} | {p:.1%} |\n"
    return out

demo = gr.Interface(
    fn=classify_post,
    inputs=gr.Textbox(lines=4, placeholder="Paste an NBA post or comment here..."),
    outputs=gr.Markdown(),
    title="TakeMeter — NBA Discourse Classifier",
    description="Classify r/nba posts as **analysis**, **hot_take**, or **reaction**.",
    examples=[
        ["Jokic's assist-to-turnover ratio this postseason is 4.8:1 — best for any center in playoff history since the merger."],
        ["LeBron is literally the GOAT and anyone who disagrees is just hating. There is no debate."],
        ["WE JUST WON THE CHAMPIONSHIP. I'm crying and I don't care who knows. 20 years of pain and we finally did it!!"],
    ]
)
demo.launch(share=True)
